# Build WRF-Hydro / CESM

Setup, build, and run the coupled WRF-Hydro + CTSM case described in the top-level
`README.md`, driven from a notebook so it can be run on
[NCAR JupyterHub](https://jupyterhub.hpc.ucar.edu) with `$SCRATCH` visible.

Two backends are supported:

| Backend | Where | What works |
|---|---|---|
| `derecho` | NCAR JupyterHub (Derecho/Casper session) | everything: `create_newcase --mach derecho`, `case.build`, `case.submit` |
| `docker` | local Mac/Linux with the `cesm-wrf-hydro` image | source checkout, module/library sanity checks, experimenting with the build |

**Docker does not run on NCAR JupyterHub.** JupyterHub sessions are ordinary
unprivileged user processes inside a PBS job; there is no Docker daemon and no root,
so `docker run` will fail there. The container runtime available on Derecho/Casper is
Apptainer, and this image would have to be converted to a `.sif` first — see the last
section. You do not need it: Derecho already provides the full compiler/ESMF stack
through `modules/gnu-cesm`, and that is the stack `--mach derecho` is configured
against. The notebook's Jupyter kernel does *not* need to be able to build anything —
every build step is shelled out to a login shell that loads the Derecho modules
itself.

## 1. Configuration

Edit this cell. Everything downstream is derived from it.

In [ ]:
import os
import shlex
import shutil
import subprocess
import sys
from pathlib import Path

# --- case settings (mirror of the top-level Makefile) ---------------------
CASE_NAME = "hydro-test"
COMPILER  = "gnu"
COMPSET   = "I2000Ctsm50NwpSpNldasWRFHydro"
RES       = "nldas2_rnldas2_mnldas2"
PROJECT   = "NWCA0002"
MACHINE   = "derecho"

# --- environment settings ------------------------------------------------
MODULE_NAME  = "gnu-cesm"      # modules/gnu-cesm/25.12.lua
DOCKER_IMAGE = "cesm-wrf-hydro"

# --- locate the repository root ------------------------------------------
REPO = Path.cwd().resolve()
while not (REPO / "src" / "ctsm").is_dir():
    if REPO == REPO.parent:
        raise RuntimeError(
            "Could not find the wrf-hydro_cesm root above %s. "
            "Start the notebook from inside the repository." % Path.cwd()
        )
    REPO = REPO.parent

# --- scratch / case directories ------------------------------------------
# On Derecho and Casper $SCRATCH is set by ncarenv. Fall back to the
# canonical path so the cell still resolves in a bare login shell.
SCRATCH = Path(os.environ.get("SCRATCH") or f"/glade/derecho/scratch/{os.environ.get('USER', '')}")
CASE_DIR = SCRATCH / "cases" / CASE_NAME

PESFILE = REPO / "src/ctsm/components/wrfhydro/src/CPL/CESM_cpl/cime_config/config_pes.xml"
SCRIPTS = REPO / "src/ctsm/cime/scripts"

print(f"repo     {REPO}")
print(f"scratch  {SCRATCH}")
print(f"case dir {CASE_DIR}")
print(f"pesfile  {PESFILE}  (exists: {PESFILE.is_file()})")

## 2. Pick a backend

`derecho` whenever `/glade` is mounted (that is, inside a JupyterHub session or on a
login node); otherwise fall back to the Docker image for local development.

In [ ]:
ON_GLADE = Path("/glade").is_dir()
HAVE_DOCKER = shutil.which("docker") is not None

if ON_GLADE:
    BACKEND = "derecho"
elif HAVE_DOCKER:
    BACKEND = "docker"
else:
    BACKEND = None

print(f"/glade mounted : {ON_GLADE}")
print(f"docker present : {HAVE_DOCKER}")
print(f"backend        : {BACKEND}")

if BACKEND == "docker":
    print(
        "\nLocal mode. `--mach derecho` needs Derecho's module stack, PBS, and the\n"
        "/glade input data, so the case steps (sections 5-9) are Derecho-only.\n"
        "Sections 3-4 (environment check) and the source update work here."
    )
elif BACKEND is None:
    raise RuntimeError("Neither /glade nor docker is available; nothing to run against.")

## 3. Command runner

Build steps run in a **fresh `bash -l` subprocess**, not in the kernel's own
environment. That matters on JupyterHub: the kernel is started inside a conda
environment (NPL or similar), and its `PYTHONPATH` / `CONDA_PREFIX` would otherwise
leak into CIME and fight with the `npl` environment that `modules/gnu-cesm` activates.
Those variables are stripped before the shell starts.

Module loading follows the README order — `module use modules`, `module purge`,
`module load gnu-cesm`.

In [ ]:
# Variables inherited from the Jupyter kernel that must not leak into the build.
_STRIP_PREFIXES = (
    "PYTHONPATH", "PYTHONHOME", "PYTHONSTARTUP",
    "CONDA_", "VIRTUAL_ENV", "_CE_",
    "JPY_", "JUPYTER_",
)

MODULE_PRELUDE = f"""
module use {shlex.quote(str(REPO / 'modules'))}
module purge
module load {MODULE_NAME}
""".strip()


def _clean_env():
    return {k: v for k, v in os.environ.items() if not k.startswith(_STRIP_PREFIXES)}


def _docker_path(path):
    """Map a host path into the container's /src mount."""
    path = Path(path).resolve()
    try:
        return "/src" if path == REPO else "/src/" + str(path.relative_to(REPO))
    except ValueError:
        raise RuntimeError(f"{path} is outside {REPO}; not visible in the container.")


def run(cmd, cwd=None, modules=True, check=True):
    """Run `cmd` in a login shell and stream its output into the notebook.

    On the derecho backend the shell is local and loads modules/gnu-cesm.
    On the docker backend the shell runs inside DOCKER_IMAGE with the repo
    bind-mounted at /src.
    """
    cwd = Path(cwd) if cwd else REPO
    script = f"set -o pipefail\n{MODULE_PRELUDE if modules and BACKEND == 'derecho' else ''}\n{cmd}"

    if BACKEND == "docker":
        # The ESMF base image sets ENTRYPOINT ["/bin/bash", "-l"], which would
        # swallow the command; override it so `-lc <script>` is what bash sees.
        argv = [
            "docker", "run", "--rm",
            "-v", f"{REPO}:/src",
            "-w", _docker_path(cwd),
            "--entrypoint", "/bin/bash",
            DOCKER_IMAGE, "-lc", script,
        ]
        env = None
    else:
        argv = ["bash", "-lc", script]
        env = _clean_env()

    print(f"$ ({BACKEND}:{cwd}) {cmd}\n", flush=True)
    proc = subprocess.Popen(
        argv, cwd=None if BACKEND == "docker" else str(cwd), env=env,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
    )
    for line in proc.stdout:
        print(line, end="", flush=True)
    rc = proc.wait()
    if check and rc != 0:
        raise RuntimeError(f"command failed with exit code {rc}: {cmd}")
    return rc

## 4. Environment check

Confirms the compilers, NetCDF/PnetCDF, ESMF, and Python that the build will actually
use. Run this first — it is cheap, and it is the step that catches a bad module
environment before a 30-minute build does.

In [ ]:
if BACKEND == "derecho":
    run("module list 2>&1; echo; which mpif90 nc-config pnetcdf_version python3; "
        "echo; echo ESMFMKFILE=$ESMFMKFILE")
else:
    run("which mpifort nc-config nf-config pnetcdf_version python3; "
        "echo; echo ESMFMKFILE=$ESMFMKFILE; nc-config --version; pnetcdf_version | head -3",
        modules=False)

## 5. Update the source tree

Off by default — it mutates the checkout. Note that CTSM's submodules are managed by
`git-fleximod`, so `git submodule update --init` must **not** be recursive.

In [ ]:
UPDATE_SOURCE = False

if UPDATE_SOURCE:
    run("git submodule update --init", cwd=REPO)
    run("./bin/git-fleximod status", cwd=REPO / "src/ctsm")
    run("./bin/git-fleximod update", cwd=REPO / "src/ctsm")
else:
    print("skipped; set UPDATE_SOURCE = True to sync submodules")

## 6. Create the case

Equivalent to `make setup`. `create_newcase` refuses to overwrite an existing case
directory, so remove it first if you are starting over.

In [ ]:
assert BACKEND == "derecho", "case creation requires the Derecho machine config and /glade"

if CASE_DIR.exists():
    print(f"{CASE_DIR} already exists -- skipping create_newcase.")
    print("To start over:  shutil.rmtree(CASE_DIR)  (and remove the bld/run dirs under $SCRATCH)")
else:
    run(
        "./create_newcase"
        f" --case {shlex.quote(str(CASE_DIR))}"
        f" --mach {MACHINE}"
        f" --compiler {COMPILER}"
        f" --compset {COMPSET}"
        f" --res {RES}"
        " --run-unsupported"
        f" --project {PROJECT}"
        f" --pesfile {shlex.quote(str(PESFILE))}",
        cwd=SCRIPTS,
    )

## 7. Configure and set up the case

One-hour run, hourly river-routing coupling, then `case.setup`.

In [ ]:
run("./xmlchange STOP_OPTION=nhours,STOP_N=1,ROF_NCPL=24", cwd=CASE_DIR)
run("./case.setup", cwd=CASE_DIR)

In [ ]:
# Optional: inspect the namelists CIME generated.
run("./preview_namelists", cwd=CASE_DIR)

## 8. Build

A CESM build is long and heavily parallel. Under JupyterHub you are already on a
compute node, so building in place is fine. From a Derecho **login** node, set
`USE_QCMD = True` to push the build onto a batch node instead.

In [ ]:
USE_QCMD = False

build_cmd = "./case.build --verbose"
if USE_QCMD:
    build_cmd = f"qcmd -A {PROJECT} -- {build_cmd}"

run(build_cmd, cwd=CASE_DIR)

## 9. First run

Per the README, the very first run must have WRF-Hydro on a single process so that
`run/DOMAIN/geo_em.d01.nc` gets created; afterwards `NTASKS_ROF` can go back to 8.
The rest of the job still uses all eight PETs.

In [ ]:
FIRST_RUN = True   # set False once run/DOMAIN/geo_em.d01.nc exists

geo_em = CASE_DIR / "run" / "DOMAIN" / "geo_em.d01.nc"
ntasks_rof = 1 if FIRST_RUN and not geo_em.is_file() else 8

print(f"geo_em.d01.nc present: {geo_em.is_file()}  ->  NTASKS_ROF={ntasks_rof}")
run(f"./xmlchange NTASKS_ROF={ntasks_rof}", cwd=CASE_DIR)
run("./case.setup --reset", cwd=CASE_DIR)
run("./xmlquery NTASKS_ROF", cwd=CASE_DIR)

In [ ]:
run("./case.submit", cwd=CASE_DIR)
# interactively instead:
# run("./case.submit --no-batch", cwd=CASE_DIR)

In [ ]:
# Job status and the tail of the run log.
run("qstat -u $USER || true", cwd=CASE_DIR, check=False)
run("ls -lt run/ | head -20", cwd=CASE_DIR, check=False)

## Containers on NCAR JupyterHub

Short answer: **no Docker.** A JupyterHub session is a PBS job running as you on a
Casper or Derecho node. Docker needs a root-owned daemon, which HPC systems do not
give users, so `docker run` will not work from a notebook there. The `docker` backend
above is for your Mac only.

If you do want a container on Derecho, the supported runtime is Apptainer:

```bash
module avail apptainer          # confirm it is there
```

Getting this image over is the awkward part — Apptainer reads a `.sif`, not your Mac's
local Docker daemon. Either push to a registry and pull on Derecho:

```bash
# on the Mac -- note --platform: Derecho is x86_64, Apple silicon is arm64
docker buildx build --platform linux/amd64 -t <registry>/cesm-wrf-hydro:latest src/docker
docker push <registry>/cesm-wrf-hydro:latest

# on Derecho
apptainer pull cesm-wrf-hydro.sif docker://<registry>/cesm-wrf-hydro:latest
apptainer exec --bind $SCRATCH cesm-wrf-hydro.sif bash -lc '...'
```

or build the `.sif` locally (`apptainer build ... docker-daemon://cesm-wrf-hydro:latest`,
needs Apptainer on the Mac, i.e. inside a Linux VM) and `scp` it over.

**But you almost certainly should not.** The container is not a shortcut around a
missing Jupyter environment, because the Jupyter environment is not what builds this
project — `case.build` does, in a subprocess, using whatever the Derecho modules
provide. Section 3's runner is exactly that. And on Derecho the container is strictly
worse: it has no `derecho` machine config, no PBS, no Cray MPI, and no `/glade`
inputdata. The `container` machine that ships in
`src/ctsm/ccs_config/machines/container/` also does not match this image — it expects
mpich under `/usr/local`, while `src/docker/Dockerfile` builds on ESMF's OpenMPI spack
view — so a full in-container case build would need a new machine config that does not
exist in this repo yet.